# YOLOv8 Nano Model Training for License Plate Detection
Run the cells below sequentially to install requirements, download your dataset, validate it, and train your model.

In [ ]:
!pip install ultralytics pyyaml kaggle torch

In [ ]:
import os
import sys
import yaml
import zipfile
import shutil
from pathlib import Path
from ultralytics import YOLO

# ============================================================================
# CONFIGURATION SECTION
# ============================================================================

# Absolute path to the dataset directory
DATASET_DIR = os.path.abspath("C:\\Users\\bss10\\Desktop\\drishti\\drishti_kavach\\archive\\IDDDetectionsYOLODataset")

# Kaggle dataset identifier
KAGGLE_DATASET = "redzapdos123/indian-driving-dataset-detections-yolov11"

# Training hyperparameters
EPOCHS = 10           # Number of training epochs
BATCH_SIZE = 8        # Batch size (CPU-friendly; use 16 if you have sufficient RAM)
IMG_SIZE = 640        # Image size for training
DEVICE = "cpu"        # Training device (CPU-only for Raspberry Pi compatibility)
MODEL = "yolov8n.pt"  # YOLOv8 nano model

In [ ]:
def dataset_exists_and_valid(dataset_path):
    required_dirs = ["train", "val", "test"]
    required_files = ["data.yaml"]
    
    if not os.path.isdir(dataset_path):
        return False
    
    for subdir in required_dirs:
        subdir_path = os.path.join(dataset_path, subdir)
        if not os.path.isdir(subdir_path):
            return False
    
    for filename in required_files:
        file_path = os.path.join(dataset_path, filename)
        if not os.path.isfile(file_path):
            return False
    
    return True

def download_dataset_from_kaggle(dataset_id, target_dir):
    try:
        import kaggle
    except ImportError:
        raise Exception("kaggle library not found. Install it with: pip install kaggle")
    
    print(f"\n📥 Downloading dataset from Kaggle: {dataset_id}")
    print(f"   Target directory: {target_dir}")
    
    os.makedirs(target_dir, exist_ok=True)
    
    try:
        kaggle.api.dataset_download_files(
            dataset_id,
            path=target_dir,
            unzip=False
        )
        print("✓ Dataset download completed.")
    except Exception as e:
        raise Exception(f"Failed to download dataset: {e}. Ensure kaggle.json is placed at ~/.kaggle/kaggle.json")

def extract_dataset(target_dir):
    zip_files = [f for f in os.listdir(target_dir) if f.endswith('.zip')]
    
    if not zip_files:
        print("⚠ No ZIP files found to extract.")
        return
    
    for zip_file in zip_files:
        zip_path = os.path.join(target_dir, zip_file)
        print(f"\n📦 Extracting: {zip_file}")
        
        try:
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(target_dir)
            
            os.remove(zip_path)
            print(f"✓ Extracted and removed: {zip_file}")
        except Exception as e:
            raise Exception(f"Failed to extract {zip_file}: {e}")

def ensure_dataset_available(dataset_dir, kaggle_dataset_id):
    if dataset_exists_and_valid(dataset_dir):
        print(f"✓ Dataset found and validated at: {dataset_dir}")
        return
    
    print(f"\n⚠ Dataset not found or invalid at: {dataset_dir}")
    print("Initiating automatic download from Kaggle...")
    
    download_dataset_from_kaggle(kaggle_dataset_id, dataset_dir)
    extract_dataset(dataset_dir)
    
    if not dataset_exists_and_valid(dataset_dir):
        raise Exception("Dataset validation failed after download.")
    
    print(f"✓ Dataset is now ready at: {dataset_dir}")

In [ ]:
def validate_dataset_structure(dataset_root):
    dataset_path = Path(dataset_root)
    
    if not dataset_path.exists():
        raise FileNotFoundError(f"Dataset root directory not found: {dataset_root}")
    
    required_dirs = [
        "train/images",
        "train/labels",
        "val/images",
        "val/labels",
        "test/images",
        "test/labels",
    ]
    
    for dir_name in required_dirs:
        dir_path = dataset_path / dir_name
        if not dir_path.exists():
            raise FileNotFoundError(f"Required directory not found: {dir_path}")
        if not list(dir_path.glob("*")):
            print(f"Warning: {dir_path} is empty")
    
    data_yaml = dataset_path / "data.yaml"
    if not data_yaml.exists():
        raise FileNotFoundError(f"data.yaml not found: {data_yaml}")
    
    print(f"✓ Dataset structure validated at: {dataset_path}")

def update_data_yaml(dataset_root):
    dataset_path = Path(dataset_root)
    data_yaml_path = dataset_path / "data.yaml"
    
    with open(data_yaml_path, "r") as f:
        data = yaml.safe_load(f)
    
    data["path"] = str(dataset_path.absolute())
    data["train"] = str((dataset_path / "train/images").absolute())
    data["val"] = str((dataset_path / "val/images").absolute())
    data["test"] = str((dataset_path / "test/images").absolute())
    
    with open(data_yaml_path, "w") as f:
        yaml.dump(data, f, default_flow_style=False)
    
    print(f"✓ data.yaml updated with absolute paths")
    return str(data_yaml_path)

In [ ]:
def train_model(data_yaml_path):
    print(f"\n{'='*70}")
    print("Starting YOLOv8 Nano Model Training")
    print(f"{'='*70}")
    print(f"Model: {MODEL}")
    print(f"Device: {DEVICE}")
    print(f"Epochs: {EPOCHS}")
    print(f"Batch Size: {BATCH_SIZE}")
    print(f"Image Size: {IMG_SIZE}x{IMG_SIZE}")
    print(f"Data Config: {data_yaml_path}")
    print(f"{'='*70}\n")
    
    model = YOLO(MODEL)
    
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        batch=BATCH_SIZE,
        imgsz=IMG_SIZE,
        device=DEVICE,
        patience=5,
        save=True,
        verbose=True,
        project="runs/detect",
        name="license_plate_detection",
    )
    
    return results

def print_training_summary(results):
    run_dir = Path("runs/detect/license_plate_detection")
    best_weights = run_dir / "weights/best.pt"
    last_weights = run_dir / "weights/last.pt"
    
    print(f"\n{'='*70}")
    print("Training Complete!")
    print(f"{'='*70}")
    
    if best_weights.exists():
        best_path = best_weights.absolute()
        print(f"✓ Best weights saved at:\n  {best_path}")
    else:
        print("⚠ Best weights file not found. Check training output above.")
    
    if last_weights.exists():
        last_path = last_weights.absolute()
        print(f"\nLast checkpoint saved at:\n  {last_path}")
    
    artifacts_path = run_dir.absolute()
    print(f"\nAll training artifacts saved at:\n  {artifacts_path}")
    print(f"{'='*70}\n")

In [6]:
try:
    print(f"\n{'='*70}")
    print("YOLOv8 Nano License Plate Detection Training Pipeline")
    print(f"{'='*70}")
    print(f"Dataset Directory: {DATASET_DIR}")
    print(f"Kaggle Dataset: {KAGGLE_DATASET}")
    print(f"{'='*70}\n")
    
    ensure_dataset_available(DATASET_DIR, KAGGLE_DATASET)
    validate_dataset_structure(DATASET_DIR)
    data_yaml_path = update_data_yaml(DATASET_DIR)
    
    results = train_model(data_yaml_path)
    print_training_summary(results)
    
except FileNotFoundError as e:
    print(f"❌ Error: {e}")
except Exception as e:
    print(f"❌ Unexpected error: {e}")
    import traceback
    traceback.print_exc()

       1/10         0G      1.618      4.557      1.219          8        640: 1% ──────────── 33/4196 4.4s/it 2:18<5:02:07


KeyboardInterrupt: 